In [ ]:
import io
import tempfile
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "..")

import yaml
from fill_my_mirror.storage import R2Client
from fill_my_mirror.blender import render_with_blender
from fill_my_mirror.projection.utils import load_rgb_image, load_binary_mask

with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)
BLENDER_PATH = Path("..") / config["blender_path"]

r2 = R2Client()

def r2_bytes(key: str) -> bytes:
    return r2._s3.get_object(Bucket=r2._bucket, Key=key)["Body"].read()

def load_r2_image(key: str) -> Image.Image:
    return Image.open(io.BytesIO(r2_bytes(key))).convert("RGB")

def reproject_frontface_culling(prefix: str) -> np.ndarray | None:
    """Re-render the reflected scene with frontface culling.

    Front-facing fragments → Transparent BSDF (invisible).
    Back-facing fragments  → textured Emission (visible).
    The rendered PNG is RGBA; we composite only the opaque (back-facing)
    pixels back onto the original image inside the mirror mask.

    Returns composited RGB array, or None if R2 files are missing.
    """
    glb_key  = f"{prefix}/reflected_scene.glb"
    npz_key  = f"{prefix}/blender_render_inputs.npz"
    img_key  = f"{prefix}/original_image.png"
    mask_key = f"{prefix}/generative_refinement_mask.png"

    for key in (glb_key, npz_key, img_key, mask_key):
        if not r2.key_exists(key):
            print(f"  Missing {key}, skipping re-projection")
            return None

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        glb_path  = tmp / "reflected_scene.glb"
        npz_path  = tmp / "blender_render_inputs.npz"
        img_path  = tmp / "original_image.png"
        mask_path = tmp / "generative_refinement_mask.png"

        glb_path.write_bytes(r2_bytes(glb_key))
        npz_path.write_bytes(r2_bytes(npz_key))
        img_path.write_bytes(r2_bytes(img_key))
        mask_path.write_bytes(r2_bytes(mask_key))

        data = np.load(npz_path)
        intrinsics  = data["intrinsics"]
        image_shape = tuple(int(x) for x in data["image_shape"])

        raw_path = tmp / "raw_render_fc.png"
        bw_path  = tmp / "bw_render_fc.png"

        render_with_blender(
            blender_path=BLENDER_PATH,
            glb_path=glb_path,
            intrinsics=intrinsics,
            image_shape=image_shape,
            output_path=raw_path,
            bw_output_path=bw_path,
            tmp_dir=tmp,
            frontface_culling=True,
        )

        # RGBA: alpha=0 where front-facing (transparent), alpha=255 where back-facing
        render_rgba = np.array(Image.open(raw_path).convert("RGBA"))
        render_rgb  = render_rgba[:, :, :3]
        alpha       = render_rgba[:, :, 3]

        original = load_rgb_image(img_path)
        mirror   = load_binary_mask(mask_path)

        # Only replace pixels that are both inside the mirror mask and opaque in the render
        visible = mirror & (alpha > 128)
        composited = original.copy()
        composited[visible] = render_rgb[visible]
        return composited

def show_pair(orig: Image.Image, fc: np.ndarray | None, title: str):
    if fc is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        ax.imshow(orig)
        ax.set_title(f"{title}\n(original only — re-projection unavailable)", fontsize=12)
        ax.axis("off")
    else:
        orig_arr = np.array(orig)
        diff = np.abs(orig_arr.astype(int) - fc.astype(int)).clip(0, 255).astype(np.uint8)
        fig, axes = plt.subplots(1, 3, figsize=(20, 7))
        for ax, img, subtitle in zip(
            axes,
            [orig_arr, fc, diff],
            ["Original (black backfaces)", "Frontface culling", "Difference"],
        ):
            ax.imshow(img)
            ax.set_title(subtitle, fontsize=12)
            ax.axis("off")
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

print("Setup complete.")

## mirrorbench_v2 — `gt_geometry` (first 50)

In [ ]:
gt_keys = sorted(
    k for k in r2.list_keys("mirrorbench_v2/gt_geometry/")
    if k.endswith("/projected_image.png")
)[:50]
print(f"Found {len(gt_keys)} images (capped at 50) under mirrorbench_v2/gt_geometry/")

for key in gt_keys:
    idx    = key.split("/")[2]
    prefix = f"mirrorbench_v2/gt_geometry/{idx}"
    orig   = load_r2_image(key)
    fc     = reproject_frontface_culling(prefix)
    show_pair(orig, fc, f"GT — idx={idx}")

## R2 — `real/estimated_geometry`

In [ ]:
r2_keys = sorted(
    k for k in r2.list_keys("real/estimated_geometry/")
    if k.endswith("/projected_image.png")
)
print(f"Found {len(r2_keys)} images under real/estimated_geometry/")

for key in r2_keys:
    idx    = key.split("/")[2]
    prefix = f"real/estimated_geometry/{idx}"
    orig   = load_r2_image(key)
    fc     = reproject_frontface_culling(prefix)
    show_pair(orig, fc, f"R2 — idx={idx}")